In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load in 

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
# Input data files are available in the "../input/" directory.
# For example, running this (by clicking run or pressing Shift+Enter) will list the files in the input directory

import os
print(os.listdir("../input"))
os.chdir("/kaggle/working/")
# Any results you write to the current directory are saved as output.


['sample_submission.csv', 'test.csv.zip', 'train.csv.zip', 'labels.csv.zip', 'train.csv', 'new-york-city-taxi-fare-prediction', 'test.csv', 'labels.csv', 'description.md', 'GCP-Coupons-Instructions.rtf', 'sample_submission.csv.zip']


In [2]:
df_train = pd.read_csv('../input/train.csv',nrows = 50000,parse_dates=["pickup_datetime"])
df_train.head()


,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,2009-06-15 17:26:21.0000001,4.5,2009-06-15 17:26:21+00:00,-73.844311,40.721319,-73.841610,40.712278,1
1,2010-01-05 16:52:16.0000002,16.9,2010-01-05 16:52:16+00:00,-74.016048,40.711303,-73.979268,40.782004,1
2,2011-08-18 00:35:00.00000049,5.7,2011-08-18 00:35:00+00:00,-73.982738,40.761270,-73.991242,40.750562,2
3,2012-04-21 04:30:42.0000001,7.7,2012-04-21 04:30:42+00:00,-73.987130,40.733143,-73.991567,40.758092,1
4,2010-03-09 07:51:00.000000135,5.3,2010-03-09 07:51:00+00:00,-73.968095,40.768008,-73.956655,40.783762,1


In [3]:
df_train.describe()
df_train.dtypes


key                               object
fare_amount                      float64
pickup_datetime      datetime64[ns, UTC]
pickup_longitude                 float64
pickup_latitude                  float64
dropoff_longitude                float64
dropoff_latitude                 float64
passenger_count                    int64
dtype: object

In [4]:
df_train = df_train[(df_train['fare_amount']>0.05) & (df_train.passenger_count>0)]
#df_train = df_train[df_train.passenger_count>0]
df_train.dropna(how = 'any', axis = 'rows',inplace=True)
print('New Size: {}'.format(len(df_train)))
df_test =  pd.read_csv('../input/test.csv',parse_dates=["pickup_datetime"])


New Size: 49825


In [5]:
mask = df_train['pickup_longitude'].between(-75, -73)
mask &= df_train['dropoff_longitude'].between(-75, -73)
mask &= df_train['pickup_latitude'].between(40, 42)
mask &= df_train['dropoff_latitude'].between(40, 42)
mask &= df_train['passenger_count'].between(0, 8)
mask &= df_train['fare_amount'].between(0, 250)

df_train = df_train[mask]


In [6]:
def distance(lat1, lon1, lat2, lon2):
    p = 0.017453292519943295 # Pi/180
    a = 0.5 - np.cos((lat2 - lat1) * p)/2 + np.cos(lat1 * p) * np.cos(lat2 * p) * (1 - np.cos((lon2 - lon1) * p)) / 2
    return 0.6213712 * 12742 * np.arcsin(np.sqrt(a)) # 2*R*asin...

df_train['distance_miles'] = distance(df_train.pickup_latitude, df_train.pickup_longitude,
                                      df_train.dropoff_latitude, df_train.dropoff_longitude)
df_train['year'] = df_train.pickup_datetime.apply(lambda t: t.year)
df_train['hour'] = df_train.pickup_datetime.apply(lambda t: t.hour)
df_test['distance_miles'] = distance(df_test.pickup_latitude, df_test.pickup_longitude,
                                      df_test.dropoff_latitude, df_test.dropoff_longitude)
df_test['hour'] = df_test.pickup_datetime.apply(lambda t: pd.to_datetime(t).hour)
df_test['year'] = df_test.pickup_datetime.apply(lambda t: pd.to_datetime(t).year)


In [7]:
features = ['year', 'hour', 'distance_miles', 'passenger_count']
X = df_train[features].values
y = df_train['fare_amount'].values
X_test = df_test[features]
df_test.head(5)


,key,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,distance_miles,hour,year
0,2010-10-01 21:26:11.0000001,2010-10-01 21:26:11+00:00,-73.983130,40.761970,-73.994386,40.749236,1,1.058853,21,2010
1,2013-10-06 01:38:00.00000083,2013-10-06 01:38:00+00:00,-73.948505,40.753977,-73.808195,40.731952,2,7.500968,1,2013
2,2012-03-30 19:13:53.0000001,2012-03-30 19:13:53+00:00,-73.973964,40.791979,-73.979018,40.785544,1,0.517284,19,2012
3,2012-02-08 02:57:23.0000001,2012-02-08 02:57:23+00:00,-73.991478,40.738907,-73.907198,40.861572,2,9.553148,2,2012
4,2013-12-13 22:56:00.000000237,2013-12-13 22:56:00+00:00,-73.986281,40.740067,-73.933927,40.856781,2,8.516409,22,2013


In [8]:
import xgboost as xgb
from bayes_opt import BayesianOptimization
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split


In [9]:
X_train, X_val, y_train, y_val = train_test_split(X,y,test_size=0.2)
X_train = xgb.DMatrix(X_train, label=y_train)
X_val = xgb.DMatrix(X_val)
def xgb_eva(max_depth,gamma,colsample_bytree):
    params = {'eval_metric': 'rmse',
              'max_depth': int(max_depth),
              'subsample': 0.8,
              'eta': 0.1,
              'gamma': gamma,
              'colsample_bytree': colsample_bytree}
    cv_result = xgb.cv(params, X_train, num_boost_round=100, nfold=3)
    return -1.0 * cv_result['test-rmse-mean'].iloc[-1]


In [10]:
xgb_bo = BayesianOptimization(xgb_eva, {'max_depth': (3, 7), 
                                        'gamma': (0, 1),
                                        'colsample_bytree': (0.3, 0.9)})
# Use the expected improvement acquisition function to handle negative numbers
# Optimally needs quite a few more initiation points and number of iterations
xgb_bo.maximize(init_points=3, n_iter=5, acq='ei')


TypeError: BayesianOptimization.maximize() got an unexpected keyword argument 'acq'

In [11]:
params = xgb_bo.res['max']['max_params']
params['max_depth'] = int(params['max_depth'])


TypeError: list indices must be integers or slices, not str

In [12]:
# Train a new model with the best parameters from the search
#model2 = xgb.train(params, X, num_boost_round=250)

# Predict on testing and training set
#y_val_pred = model2.predict(X_val)
#y_train_pred = model2.predict(X_train)

# Report testing and training RMSE
#print(np.sqrt(mean_squared_error(y_val, y_val_pred)))
#print(np.sqrt(mean_squared_error(y_train, y_train_pred)))


In [13]:
model2 = xgb.train(params, xgb.DMatrix(X,label=y), num_boost_round=250)
X_testm = xgb.DMatrix(X_test.values)
y_test = model2.predict(X_testm)


NameError: name 'params' is not defined

In [14]:
X_test.head(5)


,year,hour,distance_miles,passenger_count
0,2010,21,1.058853,1
1,2013,1,7.500968,2
2,2012,19,0.517284,1
3,2012,2,9.553148,2
4,2013,22,8.516409,2


In [15]:
sub = pd.DataFrame()
sub['key'] = df_test.key
sub['fare_amount'] = y_test
sub.to_csv('submission.csv',index=False)
#out = pd.DataFrame({'key':X_test.index,'fare_amount':y_test})
#out.to_csv('submission.csv',index=False)
sub


NameError: name 'y_test' is not defined